# Figure 5 k–m — spectral PAINT of α-synuclein aggregates with Nile Red (Ximea)

**Makes up:** Figure 5 k (wavelength render), l (six aggregates), m (aggregate wavelength vs. localisations, with the oligomer and fibril clusters).

**Data:** BioStudies `S-BIAD3210`, `2026_Multicolour_Paper/Ximea/AlphaSynPAINT/20260205_alpha_syn_NR_PLL/1nM_NR_10uM_asyn72h_2p5AuNP/50ms_70p_515_LP515_bp650_200_4/` — raw Ximea TIFFs
(8 files, ~32 GB) plus the acquisition's MicroManager metadata. The deposition holds raw
data only, so this notebook fits it first.

Position 4 of the nine acquired; the one shown in the paper. The oligomer/fibril definition in
panel m is the wavelength-independent one from `SI/FigureSI_OligomervsFibril.ipynb`.

## FAST_MODE

`FAST_MODE = True` (default) downloads and fits only the first TIFF (~4 GB), enough to run the whole
pipeline end to end, with far fewer localisations than the published figure. `FAST_MODE = False`
downloads and fits the full acquisition (~32 GB; several hours of fitting; run in `tmux`/`screen`).
Hand-picked regions are in camera pixels of the full acquisition; in `FAST_MODE` they may be sparse.

---
## 0 — Imports

In [ ]:
import json
import sys
import types
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib import colors as mcolors
import seaborn as sns
from scipy.spatial import ConvexHull
from sklearn.cluster import HDBSCAN

# _biostudies_download.py lives beside this notebook (notebooks/figures/).
sys.path.insert(0, str(Path.cwd().resolve()))
import _biostudies_download as bd

from pyS3M import IOFunctions
from pyS3M import HelperFunctions
from pyS3M import sCMOSFunctions
from pyS3M import SpectralFunctions
from pyS3M import SR_Functions
from pyS3M import PlottingBase
from pyS3M import render
from pyS3M import postprocess as _postprocess
from pyS3M import NileRedFunctions
from pyS3M import FiducialDetection
from pyS3M.DriftCorrectionFunctions import Drift_Correction_Functions
from pyS3M.LinkingFunctions import link_localisations

IO = IOFunctions.IO_Functions()
H_F = HelperFunctions.Helper_Functions()
sCMOS = sCMOSFunctions.sCMOS_Functions()
S_F = SpectralFunctions.Spectral_Funcs()
SupRes_F = SR_Functions.SuperRes_Functions()   # Ximea defaults (69 nm pixels)
plotter = PlottingBase.PublicationPlotter(dark_background=False)

FD = FiducialDetection.FiducialDetector()

print("Python:", sys.version)

---
## 1 — Configuration

In [ ]:
FAST_MODE = True   # True: first TIFF only (~4 GB, illustrative). False: all 8 TIFFs (~32 GB, the published data).

PIXEL_SIZE_NM = 69.0   # Ximea camera pixel

REPO_ROOT = Path.cwd().resolve().parents[1]   # notebooks/figures -> repo root
CALIB_DIR = REPO_ROOT / "Camera_Calibrations" / "Ximea_Camera"
OUT_ROOT  = Path("outputs") / "Figure_5k-m_AlphaSyn"   # git-ignored (bare outputs/ rule)
FIG_DIR   = OUT_ROOT / ("fast" if FAST_MODE else "full")
# Point DATA_DIR at a drive with room for the download. Keep one folder per acquisition and
# per FAST_MODE setting: the fit reads every TIFF inside it.
DATA_DIR  = OUT_ROOT / ("data_fast" if FAST_MODE else "data_full")
FIG_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

gain_map   = IO.read_tiff(CALIB_DIR / "gain.tif")
offset_map = IO.read_tiff(CALIB_DIR / "offset.tif")
variance   = IO.read_tiff(CALIB_DIR / "variance.tif")
read_noise = IO.read_tiff(CALIB_DIR / "readnoise.tif")
rqe        = IO.read_tiff(CALIB_DIR / "rqe.tif")

R, G, B, wavelength = S_F.getpixelefficiency()
pixel_QYs = np.vstack([B, G, R])

smoothing_function = types.SimpleNamespace()
smoothing_function.args = {"sigma": 1.5}
smoothing_function.extent = 1.5
smoothing_function.smoothing_function = sCMOS.gaussian_filter_stack
smoothing_function.data_arg = "image"

print(f"FAST_MODE  : {FAST_MODE}")
print(f"Data dir   : {DATA_DIR.resolve()}")
print(f"Figures dir: {FIG_DIR.resolve()}")

---
## 2 — Download the raw data from BioStudies (`S-BIAD3210`)

Downloads the first TIFF (`FAST_MODE`) or the whole series, plus the acquisition's MicroManager
`*_metadata.txt` in both cases: it holds the camera ROI the frames were cropped to, which the fit
needs to crop the calibration maps. Re-running is safe: complete files are skipped and interrupted
ones resume.

In [ ]:
REMOTE_FOLDER = "2026_Multicolour_Paper/Ximea/AlphaSynPAINT/20260205_alpha_syn_NR_PLL/1nM_NR_10uM_asyn72h_2p5AuNP/50ms_70p_515_LP515_bp650_200_4"

downloaded = bd.download_acquisition(REMOTE_FOLDER, DATA_DIR, first_file_only=FAST_MODE)

# Camera ROI of the acquisition (the fitted xc/yc are relative to its corner).
roi_x, roi_y, width, height = IO.metadata_reader_imageJ(downloaded["metadata"][0])
print(f"ROI: x={roi_x} y={roi_y}, {width} x {height} px")

---
## 3 — Spot detection and fitting

The paper's settings: pfa 1e-3, sigma 1.5, fraction_true 0.2, peak wavelength 0.65 um, 12 px ROI.
`fit_imaging_data` treats the TIFFs as one series and writes `Localisations.h5` into `DATA_DIR`.
This takes about 1 h in FAST_MODE and several hours in full. A record of what was fitted is saved beside it, so re-running the notebook
reuses the fit unless the files or settings change (or `REFIT = True`).

In [ ]:
FIT_PARAMS = dict(pfa=1e-3, sigma=1.5, fraction_true=0.2, peak_wavelength=0.65, ROI_size=12, NA=1.49)

# Quick single-frame check of the spot detection before the full fit.
fig, axs = SupRes_F.example_spots_singleframe(
    DATA_DIR, pfa=FIT_PARAMS["pfa"], sigma=FIT_PARAMS["sigma"],
    fraction_true=FIT_PARAMS["fraction_true"], frame_index=10,
    variance=variance, read_noise=read_noise, rqe=rqe, offset_map=offset_map,
    gain_map=gain_map, peak_wavelength=FIT_PARAMS["peak_wavelength"],
)
plt.show()

In [ ]:
REFIT = False

fit_file = DATA_DIR / "Localisations.h5"
fit_record_file = DATA_DIR / "Localisations_fit_record.json"
fit_record = {"images": [p.name for p in downloaded["images"]], **FIT_PARAMS}

previous = json.loads(fit_record_file.read_text()) if fit_record_file.exists() else None
if REFIT or not fit_file.exists() or previous != fit_record:
    SupRes_F.fit_imaging_data(
        DATA_DIR, smoothing_function, gain_map, offset_map, rqe, read_noise,
        variance=variance, image_type=".tif", **FIT_PARAMS,
    )
    fit_record_file.write_text(json.dumps(fit_record, indent=1))
else:
    print(f"Reusing {fit_file} (same files and settings)")

---
## 4 — Quality-control filters

The cuts used for the paper, copied from the analysis notebook (sizes in nm, converted to camera px).

In [ ]:
loc_data = pd.read_hdf(fit_file)
print(f"Raw localisations: {len(loc_data):,}")

loc_data = loc_data[loc_data["xc_err"] < 100 / PIXEL_SIZE_NM]
loc_data = loc_data[loc_data["yc_err"] < 100 / PIXEL_SIZE_NM]
loc_data = loc_data[loc_data["xc_err"] > 0]
loc_data = loc_data[loc_data["yc_err"] > 0]

loc_data = loc_data[loc_data["s_x_err"] < 100 / PIXEL_SIZE_NM]
loc_data = loc_data[loc_data["s_y_err"] < 100 / PIXEL_SIZE_NM]
loc_data = loc_data[(loc_data["s_x"] > 75 / PIXEL_SIZE_NM) & (loc_data["s_x"] < 175 / PIXEL_SIZE_NM)]
loc_data = loc_data[(loc_data["s_y"] > 75 / PIXEL_SIZE_NM) & (loc_data["s_y"] < 175 / PIXEL_SIZE_NM)]

loc_data = loc_data[loc_data["A_B_err"] < 0.01]
loc_data = loc_data[loc_data["A_G_err"] < 0.01]
loc_data = loc_data[loc_data["A_R_err"] < 0.01]

loc_data = loc_data[loc_data["A_B"] > 0.005]
loc_data = loc_data[loc_data["A_G"] > 0.05]
loc_data = loc_data[loc_data["A_R"] > 0.05]

loc_data = loc_data[loc_data["bg_B_err"] < 0.15]
loc_data = loc_data[loc_data["bg_G_err"] < 0.15]
loc_data = loc_data[loc_data["bg_R_err"] < 0.15]

loc_data = loc_data[loc_data["photons"] > 200]
loc_data = loc_data[loc_data["chi_sqr"] < 4]
print(f"After QC: {len(loc_data):,}")

---
## 5 — Drift correction (AIM) and linking

Localisations of one molecule in consecutive frames (within the median localisation error,
up to 2 dark frames) are merged into one.

In [ ]:
drift_corrector = Drift_Correction_Functions()

info = [{
    "Width": width,
    "Height": height,
    "Frames": int(loc_data["frame"].max()),
    "Pixelsize": PIXEL_SIZE_NM,
}]

corrected_locs, drift_result = drift_corrector.undrift(
    locs=loc_data.to_records(index=False),
    info=info,
    method="aim",
    segmentation=10,
    intersect_d=20 / PIXEL_SIZE_NM,
    roi_r=60 / PIXEL_SIZE_NM,
)
corrected_locs = pd.DataFrame(corrected_locs)

plt.plot(drift_result.drift_x * PIXEL_SIZE_NM, label="x")
plt.plot(drift_result.drift_y * PIXEL_SIZE_NM, label="y")
plt.xlabel("frame"); plt.ylabel("drift / nm"); plt.legend()
plt.show()

# Frame count from the fitted data, so FAST_MODE (first file only) gets the right number.
n_frames = int(loc_data["frame"].max()) + 1
link_r = 0.5 * (np.median(corrected_locs["xc_err"]) + np.median(corrected_locs["yc_err"]))
linked = link_localisations(corrected_locs, n_frames=n_frames, r_max=link_r, max_dark_time=2)
print(f"Linked localisations: {len(linked):,}")

---
## 6 — Remove fiducials

Fiducial markers bind for most of the movie. Bright, dense regions of a 1x render that hold
localisations in at least 7.5% of frames are removed.

In [ ]:
info = [{"Width": width, "Height": height, "Frames": int(linked["frame"].max()), "Pixelsize": PIXEL_SIZE_NM}]
image_1x = render.render(locs=linked.to_records(index=False), info=info, blur_method="smooth", oversampling=1)[1]

region_centers, binary_mask, threshold, _ = drift_corrector.detect_high_density_regions_from_image(
    smoothed_image=image_1x,
    histogram_bins=821,
    threshold_percentile=99.94,
    pixelsize=PIXEL_SIZE_NM,
    title="Fiducial detection",
)
selected_puncta, _ = drift_corrector.select_puncta_from_regions(
    locs=linked.to_records(index=False),
    region_centres=region_centers,
    binary_mask=binary_mask,
    pixelsize=PIXEL_SIZE_NM,
    selection_box_size_nm=500.0,
    min_localisations_per_region=int(0.075 * info[0]["Frames"]),
    title="Fiducial selection",
    create_plot=True,
)
linked_fremoved = pd.DataFrame(FD.remove_puncta_locs(linked.to_records(index=False), selected_puncta))
print(f"{len(linked) - len(linked_fremoved):,} fiducial localisations removed, {len(linked_fremoved):,} left")

IO.write_h5_database(df=linked_fremoved, filepath=DATA_DIR / "Localisations_Undrifted_Linked_FiducialsRemoved.h5",
                     normalise_photons=False)

---
## 7 — Segment aggregates and fit a wavelength per 50 nm pixel

Aggregates are regions of the 8x render above a percentile threshold, at least 5000 nm^2
and 50 localisations.

In [ ]:
seg_locs, seg_stats = _postprocess.segment_locs_by_rendered_image(
    linked_fremoved,
    width=width,
    height=height,
    oversampling=8,
    pixel_size_nm=PIXEL_SIZE_NM,
    min_area_nm2=5000.0,
    min_localisations=50,
    threshold_method="percentile",
    callback="console",
    blur_method="gaussian",
    verbose=True,
)
seg_locs = seg_locs.rename(columns={"aggregate_id": "cluster_id", "aggregate_area_nm2": "cluster_area_nm2"})
seg_stats = seg_stats.rename(columns={"aggregate_id": "cluster_id"})

clustered_file = DATA_DIR / "Localisations_Undrifted_Linked_FiducialsRemoved_clusteredlocs.h5"
IO.write_h5_database(df=seg_locs, filepath=clustered_file, append=False, normalise_photons=False)
IO.write_h5_database(df=seg_stats, filepath=DATA_DIR / "Localisations_Undrifted_Linked_FiducialsRemoved_averagedoverclusters.h5",
                     append=False, normalise_photons=False)
print(f"{seg_locs['cluster_id'].nunique()} aggregates, {len(seg_locs):,} localisations")

In [ ]:
nrf = NileRedFunctions.NileRed_Functions()
filters = [
    "semrock-ff01-650-200",
    "semrock-di03-r514-t1-25x36",
    "semrock-ff01-515-lp",
]
camera_params = {"pixel_QYs": pixel_QYs, "wavelength": wavelength}

# Fits a wavelength per 50 nm analysis pixel (at least 10 localisations each) and writes
# it back into the clustered-localisations file as wl_pixel / pixel_ix.
df_with_wavelengths, grid_info = nrf.fit_wavelengths_pixelated(
    h5_path=str(clustered_file),
    filter_names=filters,
    camera_parameters=camera_params,
    output_path=str(clustered_file),
    min_localisations=10,
    pixel_size_nm=50,
    aggregate_id_column="cluster_id",
)
print(f"Wavelength-fitted localisations: {len(df_with_wavelengths):,}")

---
## 8 — Figure 5k — wavelength render (600–645 nm)

In [ ]:
OVERSAMPLING = 8
WL_MIN, WL_MAX = 600, 645

_, image_grey, image_colour = render.render(
    locs=df_with_wavelengths.to_records(index=False),
    oversampling=OVERSAMPLING,
    viewport=((0, 0), (np.max(df_with_wavelengths["yc"]), np.max(df_with_wavelengths["xc"]))),
    blur_method="gaussian_colour",
    cparam="wl_pixel",
    c_min=WL_MIN,
    c_max=WL_MAX,
    maxdensperc=99.5,
    mindensperc=1,
    min_blur_width=4 * PIXEL_SIZE_NM / 1000,
)

fig, ax = plotter.two_column_plot(height=2, width=2)
ax = plotter.colour_image_plot(ax, image_colour, cbar="on", cbarlabel=r"$\lambda_{model}$/nm", cmap="jet",
                               c_min=WL_MIN, c_max=WL_MAX, pixelsize=PIXEL_SIZE_NM / OVERSAMPLING)
plt.savefig(FIG_DIR / "Image_AlphaSyn_Wavelength.svg", dpi=600, format="svg")
plt.show()

---
## 9 — Per-aggregate statistics

Size and shape from the localisation cloud, corrected for localisation error
(covariance minus the mean localisation variance), and a photon-weighted mean wavelength.
Same definitions as `SI/FigureSI_OligomervsFibril.ipynb`.

In [ ]:
def _convex_hull_area_nm2(xc_nm, yc_nm):
    """Convex hull area in nm^2 (NaN for fewer than 3 points or collinear ones)."""
    if len(xc_nm) < 3:
        return np.nan
    try:
        return float(ConvexHull(np.column_stack([xc_nm, yc_nm])).volume)   # .volume is the area in 2D
    except Exception:
        return np.nan


def compute_aggregate_stats(df, pixel_size_nm=PIXEL_SIZE_NM, min_locs_for_shape=10):
    """One row per aggregate: size/shape (precision-corrected), wavelength, brightness, centroid."""
    records = []
    for cid, grp in df.groupby("cluster_id"):
        xc_nm = grp["xc"].values * pixel_size_nm
        yc_nm = grp["yc"].values * pixel_size_nm
        n = len(grp)

        total_phot = grp["A_R"].values + grp["A_G"].values + grp["A_B"].values
        w = total_phot / total_phot.sum() if total_phot.sum() > 0 else np.ones(n) / n
        mean_wl = float(np.sum(w * grp["wl_pixel"].values))

        major = minor = area_ellipse = np.nan
        if n >= min_locs_for_shape:
            cov = np.cov(xc_nm, yc_nm)
            cov[0, 0] = max(cov[0, 0] - np.mean(grp["xc_err"].values ** 2) * pixel_size_nm ** 2, 0.0)
            cov[1, 1] = max(cov[1, 1] - np.mean(grp["yc_err"].values ** 2) * pixel_size_nm ** 2, 0.0)
            lam_min, lam_max = (max(float(v), 0.0) for v in np.linalg.eigvalsh(cov))
            if lam_max > 0:
                major, minor = 2 * np.sqrt(lam_max), 2 * np.sqrt(lam_min)
                area_ellipse = float(np.pi * major * minor)

        records.append({
            "cluster_id": cid, "n_locs": n, "mean_wl": mean_wl,
            "area_hull_nm2": _convex_hull_area_nm2(xc_nm, yc_nm), "area_ellipse_nm2": area_ellipse,
            "major_axis_nm": major, "minor_axis_nm": minor,
            # Median, not mean: a few extreme localisations would dominate the mean.
            "median_photon_rate": float(grp["photon_rate"].median()),
            "cx_px": float(grp["xc"].mean()), "cy_px": float(grp["yc"].mean()),
        })
    return pd.DataFrame(records)


agg_stats = compute_aggregate_stats(df_with_wavelengths)
print(f"{len(agg_stats)} aggregates")

---
## 10 — Figure 5l — six aggregates

Re-fitting renumbers the aggregates, so the published ones are found by their centroid
(camera px, read from the analysis used for the paper): the nearest new aggregate within
`MATCH_TOL_PX` is shown. Numbers are the panel labels.

In [ ]:
PANEL_L = {   # panel label: centroid (camera px) of the published aggregate
    1: (360.7, 36.5),    # long fibril
    2: (517.0, 360.2),   # fibril
    3: (424.6, 132.9),
    4: (395.7, 350.5),
    5: (204.1, 427.4),
    6: (69.7, 490.4),
}
MATCH_TOL_PX = 5.0

panel_l_ids = {}
for label, (cx, cy) in PANEL_L.items():
    d = np.hypot(agg_stats["cx_px"] - cx, agg_stats["cy_px"] - cy)
    i = int(np.argmin(d))
    if d.iloc[i] <= MATCH_TOL_PX:
        panel_l_ids[label] = agg_stats["cluster_id"].iloc[i]
        row = agg_stats.iloc[i]
        print(f"{label}: aggregate {row.cluster_id} ({d.iloc[i]:.1f} px away), "
              f"{row.n_locs} localisations, {row.mean_wl:.1f} nm")
    else:
        print(f"{label}: no aggregate within {MATCH_TOL_PX} px (nearest {d.iloc[i]:.1f} px; expected in FAST_MODE)")

In [ ]:
def plot_single_aggregate(cluster_id, df, fov_colour, fov_grey, pad_px=2, scalebar_nm=200,
                          vmin_pct=1.0, vmax_pct=99.5, box_color="white", box_lw=0.5, savepath=None):
    """Full field with a box around one aggregate (left) and a crop of the render (right).

    The crop's brightness is rescaled to its own vmin_pct-vmax_pct density percentiles,
    keeping hue (wavelength) and saturation.
    """
    grp = df[df["cluster_id"] == cluster_id]
    x_lo, x_hi = grp["xc"].min() - pad_px, grp["xc"].max() + pad_px
    y_lo, y_hi = grp["yc"].min() - pad_px, grp["yc"].max() + pad_px

    H, W = fov_colour.shape[:2]
    sx0, sx1 = max(0, int(round(x_lo * OVERSAMPLING))), min(W, int(round(x_hi * OVERSAMPLING)))
    sy0, sy1 = max(0, int(round(y_lo * OVERSAMPLING))), min(H, int(round(y_hi * OVERSAMPLING)))
    crop = fov_colour[sy0:sy1, sx0:sx1].astype(np.float32)
    crop_grey = fov_grey[sy0:sy1, sx0:sx1].astype(np.float32)

    lo, hi = np.percentile(crop_grey, vmin_pct), np.percentile(crop_grey, vmax_pct)
    if crop.max() > 1.0:
        crop = crop / crop.max()
    hsv = mcolors.rgb_to_hsv(crop)
    hsv[..., 2] = np.clip((crop_grey - lo) / max(float(hi - lo), 1e-6), 0.0, 1.0)
    crop = mcolors.hsv_to_rgb(hsv)

    sr_px_nm = PIXEL_SIZE_NM / OVERSAMPLING
    fig, (ax_fov, ax_agg) = plotter.two_column_plot(ncols=2, width_ratios=[1, 1], height=2, width=4)

    ax_fov.imshow(fov_colour, origin="lower", interpolation="bessel")
    ax_fov.add_patch(plt.Rectangle((x_lo * OVERSAMPLING, y_lo * OVERSAMPLING), (x_hi - x_lo) * OVERSAMPLING,
                                   (y_hi - y_lo) * OVERSAMPLING, linewidth=box_lw, edgecolor=box_color,
                                   facecolor="none"))
    ax_agg.imshow(crop, origin="lower", interpolation="none")

    for ax, img, length_nm, text in [(ax_fov, fov_colour, 10000, "10 µm"),
                                     (ax_agg, crop, scalebar_nm, f"{scalebar_nm} nm")]:
        ax.set_xticks([]); ax.set_yticks([])
        h, w = img.shape[:2]
        x2 = int(w * 0.90)
        x1 = int(x2 - length_nm / sr_px_nm)
        y = int(h * 0.09)
        ax.plot([x1, x2], [y, y], color="white", lw=1.5, solid_capstyle="butt")
        ax.text((x1 + x2) / 2, y + h * 0.04, text, color="white", fontsize=6, ha="center", va="bottom")

    sm = plt.cm.ScalarMappable(cmap=plt.get_cmap("jet"), norm=plt.Normalize(vmin=WL_MIN, vmax=WL_MAX))
    cbar = fig.colorbar(sm, ax=[ax_fov, ax_agg], fraction=0.018, pad=0.01, aspect=30)
    cbar.set_label("mean wavelength / nm", fontsize=7)
    cbar.ax.tick_params(labelsize=6)
    if savepath is not None:
        fig.savefig(savepath, dpi=300, bbox_inches="tight")
    return fig


for label, cid in panel_l_ids.items():
    plot_single_aggregate(cid, df_with_wavelengths, image_colour, image_grey,
                          savepath=FIG_DIR / f"Aggregate_panel{label}.svg")
    plt.show()

---
## 11 — Figure 5m — aggregate wavelength vs. number of localisations

Oligomer and fibril clusters are defined without using wavelength: HDBSCAN on
log(precision-corrected area) and log(median photon rate), standardised (as in
`SI/FigureSI_OligomervsFibril.ipynb`). The smaller-area core is the oligomers, the larger
the fibrils; aggregates HDBSCAN leaves unclassified are in neither. Each core is outlined by
its 2-sigma ellipse in this panel's axes (log10 localisations, wavelength).

In [ ]:
resolved = agg_stats[agg_stats["minor_axis_nm"] > 0].copy()
feat = np.log10(resolved[["area_ellipse_nm2", "median_photon_rate"]].to_numpy())
feat_z = (feat - feat.mean(axis=0)) / feat.std(axis=0)
resolved["hdb_label"] = HDBSCAN(min_cluster_size=12, min_samples=5).fit(feat_z).labels_

cores = sorted(set(resolved["hdb_label"]) - {-1},
               key=lambda u: resolved.loc[resolved.hdb_label == u, "area_ellipse_nm2"].mean())
groups = {}
if len(cores) >= 2:
    groups = {"oligomers": resolved[resolved.hdb_label == cores[0]],
              "fibrils": resolved[resolved.hdb_label == cores[-1]]}
else:
    print(f"HDBSCAN found {len(cores)} core(s); need 2 to mark oligomers and fibrils (expected in FAST_MODE)")
for name, g in groups.items():
    print(f"{name:9s}: n = {len(g):3d}, median wavelength {g.mean_wl.median():.1f} nm, "
          f"median {g.n_locs.median():.0f} localisations")

In [ ]:
def cov_ellipse(x, y, n_sigma=2, **kwargs):
    """n_sigma covariance ellipse of the points (x, y)."""
    vals, vecs = np.linalg.eigh(np.cov(x, y))
    angle = np.degrees(np.arctan2(vecs[1, -1], vecs[0, -1]))
    w, h = 2 * n_sigma * np.sqrt(vals[::-1])
    return patches.Ellipse((np.mean(x), np.mean(y)), w, h, angle=angle, fill=False, **kwargs)


plot_df = agg_stats.assign(log_n=np.log10(agg_stats["n_locs"]))
fig, ax = plotter.one_column_plot(width=2, height=2)
sns.kdeplot(data=plot_df, x="log_n", y="mean_wl", ax=ax, fill=True, cmap="Blues", bw_adjust=0.5, levels=20,
            cbar=True, cbar_kws={"label": "probability density"})
for name, g in groups.items():
    ax.add_patch(cov_ellipse(np.log10(g["n_locs"]), g["mean_wl"], ls="--", lw=1, color="k"))
    ax.text(np.log10(g["n_locs"]).mean(), g["mean_wl"].mean(), name, fontsize=6, ha="center", va="center")
ax.grid(True, lw=0.5, alpha=0.5, ls="--", color="grey")
ax.set_xlabel(r"log$_{10}$ $N_{molecules}$")
ax.set_ylabel(r"$\lambda_{model}$/nm")
plt.savefig(FIG_DIR / "AlphaSyn_Analysis_AggregatevsParameters.svg", dpi=600, format="svg")
plt.show()